In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 139,
 'tn': 2605,
 'fp': 32,
 'fn': 224,
 'misclassification_rate': 0.08533333333333333,
 'false_positive_rate': 0.012135001896094046,
 'false_negative_rate': 0.6170798898071626}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 39,
 'tn': 854,
 'fp': 20,
 'fn': 87,
 'misclassification_rate': 0.107,
 'false_positive_rate': 0.02288329519450801,
 'false_negative_rate': 0.6904761904761905}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I was able to get the misclassification rate of the model down to a 10.7%, down from over 12%. That is still not great odds, but still a good improvement. The false negative rate was the hardest rate to reduce, and I was only able to achive a 69.0%, still that is almost a 10% reduction, which is helpful, but it still misses more bots than it can catch. I would say that this bot predictor is best at confirming someone is human, rather than being good at catching bots. Also, since the bots are only about 12% of the actual data, the model will almost always be biased towards predicting humans, the majority of the data, and the parameters can only be tuned so much. 

In order to achieve my results, found that the max_depth was overfitting at 8, and the model was pretty much just memorizing the training data instead of learning real patterns. I started by lowering the max_depth, down to 2 to reduce that overfitting. I also lowered the subsamples to 0.8, to vary what each tree sees. I lowered the learning rate to 0.5 and the increased the n_estimators to 300. From there, I would change one value at a time and retest the data, until I found a wall of improvement, and the parameters I found to be the best fit are:
learning rate - 1.0
n_estimators - 200
max_depth - 2
subsample - 0.8

Doing this made the model see a lot of smaller trees rather than just a few complicated ones. 

### What are potential ramifications of false positives from the model?

A false positive means that a real human can get flagged as a bot. The ramifications depedn on how the information is ebing used. If a human gets flagged as a bot, are they facing a trial, a simple accoutn suspesnion, or being immediately executed without question in a post-apocalyptic sci-fi world? A simple double check with other means could confirm a human in this world fairly quickly. 

### What are potential ramifications of false negatives from the model?

This would be the most serious ramifications, as missing a potential bot is the reason this bot predictor model exists. It could allow a bot to cause an actual person harm, if they are connected to the internet in order to collect data, create spam or viruses, or even steal someon's identity. Unfortunately, at a 69% false negative rate, most bots are probably slipping by this predictor. 